> **About this notebook**
>
> This notebook is part of the example gallery of [CLABTOOLKIT](https://github.com/connectomicslab/clabtoolkit), an open-source Python toolkit for routine processing, manipulation and visualization of neuroimaging data. It shows how to use the `dicomtools` module.
>
> 📦 **Install:** `pip install clabtoolkit` ([PyPI](https://pypi.org/project/clabtoolkit/)) · 📖 **Docs:** [clabtoolkit.readthedocs.io](https://clabtoolkit.readthedocs.io) · 📄 **Paper:** [arXiv:2607.02638](https://arxiv.org/abs/2607.02638)
>
> **Author:** Yasser Alemán-Gómez (Connectomics Lab, Radiology Department, CHUV – Lausanne University Hospital)

## Testing Methods in the `DICOMTOOLS` Module

This Jupyter Notebook serves as a platform to rigorously test the various methods included in the Python module `DICOMTOOLS`.

This module is particularly valuable for researchers handling raw scanner data: organizing DICOM files
into a subject/session/series hierarchy, converting them to BIDS, naming sessions and series from the
DICOM metadata, compressing and uncompressing session folders, and extracting metadata from individual
DICOM files.

The tests are organized into the following sections for clarity and ease of navigation:

### Table of Contents

---

* 🔷 1. [`Organizing and Copying DICOM Files`](#dicom-organize). Methods dedicated to organize and copy DICOM files
    * 🔸 1.1. [`org_conv_dicoms`](#dicom-org_conv_dicoms). Organize the DICOM files in sessions and series. It can also use the demographics file to define the session ID
    * 🔸 1.2. [`organize_dicom_files`](#dicom-organize_dicom_files). Alias for org_conv_dicoms kept for backward compatibility
    * 🔸 1.3. [`org_dicom_folder`](#dicom-org_dicom_folder). Organize DICOM files into a subject/session/series folder hierarchy
    * 🔸 1.4. [`copy_dicom_file`](#dicom-copy_dicom_file). Copy the DICOM files to the output directory
    * 🔸 1.5. [`create_session_series_names`](#dicom-create_session_series_names). Create the session and series names from a DICOM object

* 🔷 2. [`Compressing and Uncompressing Sessions`](#dicom-compression). Methods dedicated to compress and uncompress the session folders containing the DICOM files
    * 🔸 2.1. [`uncompress_dicom_session`](#dicom-uncompress_dicom_session). Uncompress the session folders containing the DICOM files for all the series
    * 🔸 2.2. [`compress_dicom_session`](#dicom-compress_dicom_session). Compress the session folders containing DICOM files into tar.gz archives

* 🔷 3. [`Auxiliary Methods`](#dicom-auxiliary). Auxiliary methods to inspect DICOM files and to follow the progress of long tasks
    * 🔸 3.1. [`get_dicom_info`](#dicom-get_dicom_info). Extract the metadata from a DICOM file and return it as a dictionary

---

---
## <a id="dicom-organize"></a>1. Organizing and Copying DICOM Files
Methods dedicated to organize and copy DICOM files.

The examples use a small **synthetic** dataset written by the next cell: two subjects, three sessions and seven series of DICOM files with the header fields used by `dicomtools`. Run it before any other example.

In [ ]:
################## Synthetic DICOM dataset used by all the examples ##################
# Run this cell first. It writes unorganized DICOM files for two subjects:
#   - sub-01: two sessions (baseline 2024-03-12 09:15:30 and follow-up 2024-09-20 14:02:10)
#             with 3 and 2 series; all the files are mixed in a single flat folder.
#   - sub-02: one session (2024-04-02 10:40:00) with 2 series, exported in nested folders.
# Each series has a few 32 x 32 slices with the header fields used by dicomtools
# (PatientID, StudyDate, StudyTime, SeriesNumber, SeriesDescription, ProtocolName).
import os
import shutil

import numpy as np

import pydicom
from pydicom.dataset import Dataset, FileMetaDataset
from pydicom.uid import ExplicitVRLittleEndian, MRImageStorage, generate_uid

# Synthetic acquisition plan: {folder: [(PatientID, StudyDate, StudyTime, [(SeriesNumber, SeriesDescription, n_slices)])]}
# sub-01 was scanned twice (baseline and follow-up), sub-02 once
PLAN = {
    "sub-01": [
        ("sub-01", "20240312", "091530", [(1, "T1w_MPRAGE", 6), (2, "DWI_b1000_30dir", 4), (3, "rs-fMRI_BOLD", 4)]),
        ("sub-01", "20240920", "140210", [(1, "T1w_MPRAGE", 6), (2, "FLAIR_3D", 5)]),
    ],
    "sub-02": [
        ("sub-02", "20240402", "104000", [(1, "T1w_MPRAGE", 6), (2, "DWI_b1000_30dir", 4)]),
    ],
}


def write_slice(path, patient_id, study_date, study_time, study_uid, series_uid, ser_num, ser_desc, inst, n_slices):
    """Write one 32 x 32 MR slice with the header fields used by dicomtools."""
    meta = FileMetaDataset()
    meta.MediaStorageSOPClassUID = MRImageStorage
    meta.MediaStorageSOPInstanceUID = generate_uid()
    meta.TransferSyntaxUID = ExplicitVRLittleEndian

    ds = Dataset()
    ds.file_meta = meta
    ds.SOPClassUID = MRImageStorage
    ds.SOPInstanceUID = meta.MediaStorageSOPInstanceUID
    ds.PatientID, ds.PatientName = patient_id, f"Synthetic^{patient_id}"
    ds.PatientSex, ds.PatientBirthDate = "O", "19900101"
    ds.StudyDate = ds.SeriesDate = ds.AcquisitionDate = study_date
    ds.StudyTime = ds.SeriesTime = study_time
    ds.StudyInstanceUID, ds.SeriesInstanceUID = study_uid, series_uid
    ds.Modality, ds.Manufacturer = "MR", "SyntheticScanner"
    ds.SeriesNumber, ds.InstanceNumber = ser_num, inst
    ds.SeriesDescription = ds.ProtocolName = ser_desc
    ds.ImagePositionPatient = [0.0, 0.0, float(inst)]
    ds.ImageOrientationPatient = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0]
    ds.PixelSpacing, ds.SliceThickness = [1.0, 1.0], 1.0
    ds.Rows = ds.Columns = 32
    ds.SamplesPerPixel, ds.PhotometricInterpretation = 1, "MONOCHROME2"
    ds.BitsAllocated, ds.BitsStored, ds.HighBit, ds.PixelRepresentation = 16, 12, 11, 0
    yy, xx = np.mgrid[:32, :32]
    img = 1000 * np.exp(-((xx - 16) ** 2 + (yy - 16) ** 2) / (2 * (6 + inst) ** 2)) + 100 * ser_num
    ds.PixelData = img.astype(np.uint16).tobytes()
    ds.save_as(path, enforce_file_format=True)


def simulate_dicom_dataset(root):
    """Write unorganized DICOMs for two subjects. sub-01 files are flat, sub-02 files are nested."""
    shutil.rmtree(root, ignore_errors=True)
    rng = np.random.default_rng(0)
    for folder, studies in PLAN.items():
        for patient_id, sdate, stime, series in studies:
            study_uid = generate_uid()
            for ser_num, ser_desc, n_slices in series:
                series_uid = generate_uid()
                for inst in range(1, n_slices + 1):
                    # Scanner-like file names; sub-02 mimics an export with nested folders
                    name = f"IM_{rng.integers(1e8):08d}.dcm"
                    sub_dir = os.path.join(root, folder) if folder == "sub-01" else \
                        os.path.join(root, folder, "DICOM", f"ST{sdate[-4:]}", f"SE{ser_num:04d}")
                    os.makedirs(sub_dir, exist_ok=True)
                    write_slice(os.path.join(sub_dir, name), patient_id, sdate, stime,
                                study_uid, series_uid, ser_num, ser_desc, inst, n_slices)
    return root


def show_tree(root, max_files=2):
    """Print a folder tree, listing at most max_files files per folder."""
    for dirpath, dirnames, filenames in sorted(os.walk(root)):
        dirnames.sort()
        depth = dirpath[len(root):].count(os.sep)
        print("  " * depth + os.path.basename(dirpath) + "/")
        files = sorted(filenames)
        for f in files[:max_files]:
            print("  " * (depth + 1) + f)
        if len(files) > max_files:
            print("  " * (depth + 1) + f"... ({len(files)} files)")


EX_DIR = "/tmp/dicomtools_examples"
RAW_DIR = simulate_dicom_dataset(os.path.join(EX_DIR, "raw"))   # Absolute paths are required

def fresh_dir(name):
    """Empty output folder for one example, so every cell can be re-run."""
    path = os.path.join(EX_DIR, name)
    shutil.rmtree(path, ignore_errors=True)
    return path

print("Unorganized input folder:")
show_tree(RAW_DIR, max_files=2)


### 1.1 `org_conv_dicoms`
<a id="dicom-org_conv_dicoms"></a>

Organize the DICOM files in sessions and series. It can also use the demographics file to define the session ID.

In [ ]:
########################### Testing org_conv_dicoms ###########################
import clabtoolkit.dicomtools as cltdicom
import pandas as pd
import os

# NOTE: Run the setup cell of section 1 first (it defines RAW_DIR, fresh_dir and show_tree).

print("Test 1: Organizing all the subjects into <subject>/<session>/<series> folders...")
print("NOTE: Each folder starting with 'sub-' is a subject. The session is named after StudyDate + StudyTime")
print("      and the series after SeriesNumber + SeriesDescription.")
out_dir = fresh_dir("org_conv")
cltdicom.org_conv_dicoms(RAW_DIR, out_dir, nthreads=1)
show_tree(out_dir, max_files=0)
print("-------------------------------")
print(" ")

print("Test 2: Using a demographics table to append the visit ID to each session...")
print("NOTE: The table needs the columns participant_id, session_id and acq_date (MM/DD/YYYY).")
demog_file = os.path.join(EX_DIR, "demographics.csv")
pd.DataFrame({
    "participant_id": ["sub-01", "sub-01", "sub-02"],
    "session_id": ["V1", "V2", "V1"],
    "acq_date": ["03/12/2024", "09/20/2024", "04/02/2024"],
}).to_csv(demog_file, index=False)
out_dir = fresh_dir("org_conv_demog")
cltdicom.org_conv_dicoms(RAW_DIR, out_dir, demog_file=demog_file, nthreads=1)
for subj in sorted(os.listdir(out_dir)):
    print(f"  {subj}: {sorted(os.listdir(os.path.join(out_dir, subj)))}")
print("-------------------------------")
print(" ")

print("Test 3: Processing only some subjects (IDs given without 'sub-') with 2 threads...")
out_dir = fresh_dir("org_conv_ids")
cltdicom.org_conv_dicoms(RAW_DIR, out_dir, ids_file="02", nthreads=2)
print(f"  Subjects organized: {sorted(os.listdir(out_dir))}")
print("-------------------------------")
print(" ")

print("Test 4: Forcing the session label with ses_id (e.g. a single-session study)...")
print("NOTE: All the series of a subject go to 'ses-<ses_id>', whatever their acquisition date.")
out_dir = fresh_dir("org_conv_sesid")
cltdicom.org_conv_dicoms(RAW_DIR, out_dir, ses_id="baseline", ids_file="sub-02", nthreads=1)
show_tree(out_dir, max_files=0)
print("-------------------------------")
print(" ")

print("Test 5: Organizing and compressing each session into a tar.gz archive...")
out_dir = fresh_dir("org_conv_comp")
cltdicom.org_conv_dicoms(RAW_DIR, out_dir, boolcomp=True, nthreads=1)
print(f"  sub-01: {sorted(os.listdir(os.path.join(out_dir, 'sub-01')))}")
print("-------------------------------")


### 1.2 `organize_dicom_files`
<a id="dicom-organize_dicom_files"></a>

Alias for org_conv_dicoms kept for backward compatibility.

In [ ]:
########################### Testing organize_dicom_files ###########################
import clabtoolkit.dicomtools as cltdicom
import os

# NOTE: Run the setup cell of section 1 first (it defines RAW_DIR, fresh_dir and show_tree).

print("Test 1: Default behaviour, the same as org_conv_dicoms (subjects taken from the 'sub-' folders)...")
out_dir = fresh_dir("organize_subfolders")
cltdicom.organize_dicom_files(RAW_DIR, out_dir, nthreads=1)
print(f"  Subjects: {sorted(os.listdir(out_dir))}")
print("-------------------------------")
print(" ")

print("Test 2: With no_sub_folder=True it calls org_dicom_folder (subjects taken from the PatientID)...")
out_dir = fresh_dir("organize_patientid")
cltdicom.organize_dicom_files(RAW_DIR, out_dir, no_sub_folder=True, nthreads=1)
print(f"  Subjects: {sorted(os.listdir(out_dir))}")
print("-------------------------------")


### 1.3 `org_dicom_folder`
<a id="dicom-org_dicom_folder"></a>

Organize DICOM files into a subject/session/series folder hierarchy.

In [ ]:
########################### Testing org_dicom_folder ###########################
import clabtoolkit.dicomtools as cltdicom
import os
import shutil

# NOTE: Run the setup cell of section 1 first (it defines RAW_DIR, fresh_dir and show_tree).

print("Test 1: Organizing a folder with DICOMs of any subject, named after the PatientID tag...")
print("NOTE: The input folder is searched recursively and does not need one folder per subject.")
mixed_dir = fresh_dir("mixed_export")
for root, _, files in os.walk(RAW_DIR):          # Put all the files of both subjects in one folder
    for f in files:
        os.makedirs(mixed_dir, exist_ok=True)
        shutil.copy(os.path.join(root, f), mixed_dir)
print(f"  Files in the mixed folder: {len(os.listdir(mixed_dir))}")
out_dir = fresh_dir("org_folder")
cltdicom.org_dicom_folder(mixed_dir, out_dir, nthreads=1)
show_tree(out_dir, max_files=0)
print("-------------------------------")
print(" ")

print("Test 2: Using several worker processes. Files that are not DICOMs are reported and skipped...")
with open(os.path.join(mixed_dir, "README.txt"), "w") as f:
    f.write("Exported from the scanner")
out_dir = fresh_dir("org_folder_parallel")
cltdicom.org_dicom_folder(mixed_dir, out_dir, nthreads=2)
print("-------------------------------")
print(" ")

print("Test 3: An input folder that does not exist...")
try:
    cltdicom.org_dicom_folder(os.path.join(EX_DIR, "missing"), out_dir)
except FileNotFoundError as e:
    print(f"  FileNotFoundError: {e}")
print("-------------------------------")


### 1.4 `copy_dicom_file`
<a id="dicom-copy_dicom_file"></a>

Copy the DICOM files to the output directory.

In [ ]:
########################### Testing copy_dicom_file ###########################
import clabtoolkit.dicomtools as cltdicom
import os

# NOTE: Run the setup cell of section 1 first (it defines RAW_DIR, fresh_dir and show_tree).

dicom_files = sorted(os.path.join(RAW_DIR, "sub-01", f) for f in os.listdir(os.path.join(RAW_DIR, "sub-01")))

print("Test 1: Copying one DICOM file into its <subject>/<session>/<series> folder...")
out_dir = fresh_dir("copy")
dest_dir = cltdicom.copy_dicom_file(dicom_files[0], "sub-01", out_dir)
print(f"  Destination: {os.path.relpath(dest_dir, out_dir)}")
print(f"  Files: {os.listdir(dest_dir)}")
print("-------------------------------")
print(" ")

print("Test 2: Copying all the files of a subject one by one (what org_conv_dicoms does internally)...")
dest_dirs = {cltdicom.copy_dicom_file(f, "sub-01", out_dir) for f in dicom_files}
for d in sorted(dest_dirs):
    print(f"  {os.path.relpath(d, out_dir)}: {len(os.listdir(d))} files")
print("-------------------------------")
print(" ")

print("Test 3: Without overwrite an existing copy is left untouched; with overwrite=True it is replaced...")
copied = os.path.join(dest_dir, os.path.basename(dicom_files[0]))
with open(copied, "wb") as f:
    f.write(b"damaged copy")                              # Simulate a damaged copy
cltdicom.copy_dicom_file(dicom_files[0], "sub-01", out_dir)
print(f"  overwrite=False: damaged copy kept  -> {os.path.getsize(copied)} bytes")
cltdicom.copy_dicom_file(dicom_files[0], "sub-01", out_dir, overwrite=True)
print(f"  overwrite=True : copied again       -> {os.path.getsize(copied)} bytes (original: {os.path.getsize(dicom_files[0])})")
print("-------------------------------")
print(" ")

print("Test 4: Files that are not DICOMs are reported and skipped (None is returned)...")
not_dicom = os.path.join(EX_DIR, "notes.txt")
with open(not_dicom, "w") as f:
    f.write("Exported from the scanner")
print(f"  Returned: {cltdicom.copy_dicom_file(not_dicom, 'sub-01', out_dir)}")
print("-------------------------------")


### 1.5 `create_session_series_names`
<a id="dicom-create_session_series_names"></a>

Create the session and series names from a DICOM object.

In [ ]:
########################### Testing create_session_series_names ###########################
import clabtoolkit.dicomtools as cltdicom
import pydicom
import os

# NOTE: Run the setup cell of section 1 first (it defines RAW_DIR).

t1_file = os.path.join(RAW_DIR, "sub-02", "DICOM", "ST0402", "SE0001", sorted(os.listdir(os.path.join(RAW_DIR, "sub-02", "DICOM", "ST0402", "SE0001")))[0])

print("Test 1: Session and series names of a DICOM file...")
ds = pydicom.dcmread(t1_file)
print(f"  StudyDate={ds.StudyDate}, StudyTime={ds.StudyTime}, SeriesNumber={ds.SeriesNumber}, SeriesDescription={ds.SeriesDescription}")
print(f"  -> {cltdicom.create_session_series_names(ds)}")
print("-------------------------------")
print(" ")

print("Test 2: Unwanted characters are removed and spaces, '/' and '_' become '-'...")
ds.SeriesDescription = "T1w (MPRAGE) 1mm/iso*"
print(f"  '{ds.SeriesDescription}' -> {cltdicom.create_session_series_names(ds)[1]}")
print("-------------------------------")
print(" ")

print("Test 3: Without SeriesDescription the ProtocolName is used...")
del ds.SeriesDescription
ds.ProtocolName = "t1_mprage_sag_p2_iso"
print(f"  -> {cltdicom.create_session_series_names(ds)[1]}")
print("-------------------------------")
print(" ")

print("Test 4: StudyTime formats (fractional seconds, HHMM, missing) and a file without SeriesNumber...")
for study_time in ["091530.250000", "0915", ""]:
    ds.StudyTime = study_time
    print(f"  StudyTime={study_time!r:16} -> session {cltdicom.create_session_series_names(ds)[0]}")
del ds.SeriesNumber
print(f"  Without SeriesNumber -> series {cltdicom.create_session_series_names(ds)[1]}")
print("-------------------------------")


---
## <a id="dicom-compression"></a>2. Compressing and Uncompressing Sessions
Methods dedicated to compress and uncompress the session folders containing the DICOM files.

### 2.1 `uncompress_dicom_session`
<a id="dicom-uncompress_dicom_session"></a>

Uncompress the session folders containing the DICOM files for all the series.

In [ ]:
########################### Testing uncompress_dicom_session ###########################
import clabtoolkit.dicomtools as cltdicom
import os

# NOTE: Run the setup cell of section 1 first (it defines RAW_DIR, fresh_dir and show_tree).

# Organized and compressed dataset: <subject>/<session>.tar.gz
out_dir = fresh_dir("uncompress")
cltdicom.org_conv_dicoms(RAW_DIR, out_dir, boolcomp=True, nthreads=1)
print(f"Compressed sub-01: {sorted(os.listdir(os.path.join(out_dir, 'sub-01')))}")
print(" ")

print("Test 1: Uncompressing the sessions of one subject and keeping the tar.gz files...")
failed = cltdicom.uncompress_dicom_session(out_dir, subj_ids=["sub-02"])
print(f"  Failed: {failed}, sub-02: {sorted(os.listdir(os.path.join(out_dir, 'sub-02')))}")
print("-------------------------------")
print(" ")

print("Test 2: Uncompressing all the subjects and removing the tar.gz files...")
failed = cltdicom.uncompress_dicom_session(out_dir, boolrmtar=True)
print(f"  Failed: {failed}, sub-01: {sorted(os.listdir(os.path.join(out_dir, 'sub-01')))}")
print("-------------------------------")
print(" ")

print("Test 3: Taking the subject IDs from a text file...")
cltdicom.compress_dicom_session(out_dir, subj_ids=["sub-01"])
ids_file = os.path.join(EX_DIR, "subjects.txt")
with open(ids_file, "w") as f:
    f.write("sub-01\n")
failed = cltdicom.uncompress_dicom_session(out_dir, subj_ids=ids_file, boolrmtar=True)
print(f"  Failed: {failed}, sub-01: {sorted(os.listdir(os.path.join(out_dir, 'sub-01')))}")
print("-------------------------------")


### 2.2 `compress_dicom_session`
<a id="dicom-compress_dicom_session"></a>

Compress the session folders containing DICOM files into tar.gz archives.

In [ ]:
########################### Testing compress_dicom_session ###########################
import clabtoolkit.dicomtools as cltdicom
import tarfile
import os

# NOTE: Run the setup cell of section 1 first (it defines RAW_DIR, fresh_dir and show_tree).

out_dir = fresh_dir("compress")
cltdicom.org_conv_dicoms(RAW_DIR, out_dir, nthreads=1)

print("Test 1: Compressing the sessions of one subject but keeping the original folders...")
failed = cltdicom.compress_dicom_session(out_dir, subj_ids=["sub-02"], remove_original=False)
print(f"  Failed: {failed}, sub-02: {sorted(os.listdir(os.path.join(out_dir, 'sub-02')))}")
print("-------------------------------")
print(" ")

print("Test 2: Compressing all the subjects (the session folders are replaced by the archives)...")
print("NOTE: Sessions that already have a tar.gz file are skipped.")
failed = cltdicom.compress_dicom_session(out_dir)
print(f"  sub-01: {sorted(os.listdir(os.path.join(out_dir, 'sub-01')))}")
print("-------------------------------")
print(" ")

print("Test 3: Each archive contains the session folder with all its series...")
tar_path = os.path.join(out_dir, "sub-01", sorted(f for f in os.listdir(os.path.join(out_dir, "sub-01")) if f.endswith(".tar.gz"))[-1])
with tarfile.open(tar_path) as tar:
    members = tar.getnames()
print(f"  {os.path.basename(tar_path)}: {len(members)} members, series: {sorted({m.split('/')[1] for m in members if m.count('/') == 1})}")
print("-------------------------------")


---
## <a id="dicom-auxiliary"></a>3. Auxiliary Methods
Auxiliary methods to inspect DICOM files and to follow the progress of long tasks.

### 3.1 `get_dicom_info`
<a id="dicom-get_dicom_info"></a>

Extract the metadata from a DICOM file and return it as a dictionary.

In [ ]:
########################### Testing get_dicom_info ###########################
import clabtoolkit.dicomtools as cltdicom
import os

# NOTE: Run the setup cell of section 1 first (it defines RAW_DIR).

dicom_file = os.path.join(RAW_DIR, "sub-01", sorted(os.listdir(os.path.join(RAW_DIR, "sub-01")))[0])

print("Test 1: Extracting all the metadata of a DICOM file...")
info = cltdicom.get_dicom_info(dicom_file)
print(f"  {len(info)} tags, e.g. {list(info)[:6]}")
print("-------------------------------")
print(" ")

print("Test 2: Extracting only some tags (a single tag can also be given as a string)...")
print(f"  {cltdicom.get_dicom_info(dicom_file, tags=['PatientID', 'StudyDate', 'SeriesDescription', 'Manufacturer'])}")
print(f"  {cltdicom.get_dicom_info(dicom_file, tags='Modality')}")
print("-------------------------------")
print(" ")

print("Test 3: Handling tags that are not in the file ('warn', 'ignore' or 'raise')...")
print(f"  ignore: {cltdicom.get_dicom_info(dicom_file, tags=['PatientID', 'EchoTime'], missing_tag_behavior='ignore')}")
try:
    cltdicom.get_dicom_info(dicom_file, tags=["EchoTime"], missing_tag_behavior="raise")
except AttributeError as e:
    print(f"  raise : AttributeError: {e}")
print("-------------------------------")
print(" ")

print("Test 4: Reading a file that is not a DICOM...")
not_dicom = os.path.join(EX_DIR, "notes.dcm")
with open(not_dicom, "w") as f:
    f.write("not a DICOM file")
try:
    cltdicom.get_dicom_info(not_dicom)
except Exception as e:
    print(f"  {type(e).__name__}: {e}")
print("-------------------------------")
